In [ ]:
import pandas as pd
from google.colab import drive

In [ ]:
#drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# DEFINIÇÃO DOS CAMINHOS
CAMINHO_PNS_TXT = "data/PNS_2019_reduzido.gsheet"
CAMINHO_PNS_TRATADO = "data/PNS_2019_PREVALENCIA_FINAL.csv"


In [ ]:
# Caminhos da Base Mestra (Educação)
CAMINHO_MESTRE = "results/DADOS_MESTRE_PRONTOS_PARA_MODELAGEM.csv"
CAMINHO_FINAL_COM_SAUDE = "results/DADOS_MESTRE_COMPLETO_PRONTO.csv"

In [ ]:
#CONFIGURAÇÃO FWF PNS 2019
# Posições para (ID_MUNICIPIO, IDADE, TDAH, TEA, PESO_AMOSTRAL)
colunas_datasus = [
    (11, 16), (119, 121), (1717, 1718), (1719, 1720), (2003, 2019)
]
nomes_colunas = ['ID_MUNICIPIO', 'IDADE', 'TDAH', 'TEA', 'PESO_AMOSTRAL_PNS']

print("Iniciando Fase 5: Processamento e Merge Final...")

try:
    # 1. Processamento da Saúde (PNS FWF)
    print("1. Lendo e processando o arquivo PNS (Saúde) em formato FWF...")

    # *** A CHAVE É USAR pd.read_fwf ***
    df_pns = pd.read_fwf(
        CAMINHO_PNS_TXT,
        colspecs=colunas_datasus,
        names=nomes_colunas,
        encoding='latin-1',
        skiprows=1 # Ignora a primeira linha
    )

    print(f" PNS lido com sucesso. Total de registros: {df_pns.shape[0]}.")

# Cálculo da Prevalência
    df_pns = df_pns[(df_pns['IDADE'] >= 6) & (df_pns['IDADE'] <= 17)].copy()
    df_pns.loc[:, 'TEA'] = df_pns['TEA'].map({1: 1, 2: 0}).fillna(pd.NA)
    df_pns.loc[:, 'TDAH'] = df_pns['TDAH'].map({1: 1, 2: 0}).fillna(pd.NA)

    df_prevalencia = df_pns.groupby('ID_MUNICIPIO').apply(
        lambda x: pd.Series({
            'PREVALENCIA_TEA_PNS': (x['TEA'].fillna(0) * x['PESO_AMOSTRAL_PNS']).sum() / x['PESO_AMOSTRAL_PNS'].sum(),
            'PREVALENCIA_TDAH_PNS': (x['TDAH'].fillna(0) * x['PESO_AMOSTRAL_PNS']).sum() / x['PESO_AMOSTRAL_PNS'].sum()
        })
    ).reset_index()

    df_prevalencia['ID_MUNICIPIO'] = df_prevalencia['ID_MUNICIPIO'].astype(str).str.zfill(7).astype('Int64', errors='ignore')
    df_prevalencia.to_csv(CAMINHO_PNS_TRATADO, index=False)
    print(f" Prevalência calculada para {df_prevalencia.shape[0]} municípios.")

    # 2. Merge Final (Anexar Saúde à Base Mestra de Educação)
    print("2. Iniciando Merge Final com a Base Mestra (Educação)...")
    df_mestre = pd.read_csv(CAMINHO_MESTRE)
    df_mestre['ID_MUNICIPIO'] = df_mestre['ID_MUNICIPIO'].astype('Int64')

    df_master_completo = pd.merge(
        df_mestre,
        df_prevalencia[['ID_MUNICIPIO', 'PREVALENCIA_TEA_PNS', 'PREVALENCIA_TDAH_PNS']],
        on='ID_MUNICIPIO',
        how='left'
    )

 # 3. Exportação
    df_master_completo.to_csv(CAMINHO_FINAL_COM_SAUDE, index=False)

    print(f"\n PROJETO DE DADOS CONCLUÍDO!")
    print(f"A BASE MESTRA FINAL (Educação + Contexto + Saúde) está salva em: {CAMINHO_FINAL_COM_SAUDE}")
    print(f"Colunas Finais: {df_master_completo.shape[1]}") # Deve ser 192 (190 + 2 novas)


except FileNotFoundError as e:
    print(f"\n ERRO CRÍTICO: Arquivo não encontrado. Verifique o caminho. Detalhe: {e}")
except Exception as e:
    print(f"\n ERRO FATAL no processamento: {e}")

Iniciando Fase 5: Processamento e Merge Final...
1. Lendo e processando o arquivo PNS (Saúde) em formato FWF...

❌ ERRO CRÍTICO: Arquivo não encontrado. Verifique o caminho. Detalhe: [Errno 2] No such file or directory: '/content/drive/MyDrive/Iniciacao_cientifica/PNS_2019_reduzido.gsheet'
